In [7]:
# ============================================================
# CERTIFICADOS PARA OUVINTES — GOOGLE COLAB
# CC0 1.0 Universal
# ============================================================

!pip -q install reportlab pypdf openpyxl pandas

import io
import re
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
from google.colab import files
from pypdf import PdfReader, PdfWriter

from reportlab.pdfgen import canvas
from reportlab.platypus import Paragraph
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_CENTER


# ============================================================
# CONFIGURAÇÕES
# ============================================================

COL_NOME = "Nome"
COL_EMAIL = "E-mail"

OUTPUT_PDF = "certificados_ouvintes.pdf"
OUTPUT_EMAILS = "emails_ouvintes.txt"

BASE_FONT = 14
MIN_FONT = 10

FONTE_NORMAL = "Helvetica"
FONTE_BOLD = "Helvetica-Bold"

# AJUSTE DA CAIXA DE TEXTO
# Esses valores funcionam como ponto de partida.
# Se o texto ficar muito alto/baixo, ajuste principalmente TEXT_BOX_Y.
TEXT_BOX_WIDTH = 430
TEXT_BOX_HEIGHT = 120
TEXT_BOX_Y = 300


# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def limpar_texto(valor):
    if pd.isna(valor) or valor is None:
        return ""
    return re.sub(r"\s+", " ", str(valor)).strip()


def escapar_html(valor):
    return (
        valor.replace("&", "&amp;")
             .replace("<", "&lt;")
             .replace(">", "&gt;")
    )


def normalizar_email(email):
    return limpar_texto(email).lower()


def data_geracao_extenso():
    meses = {
        1: "JANEIRO", 2: "FEVEREIRO", 3: "MARÇO", 4: "ABRIL",
        5: "MAIO", 6: "JUNHO", 7: "JULHO", 8: "AGOSTO",
        9: "SETEMBRO", 10: "OUTUBRO", 11: "NOVEMBRO", 12: "DEZEMBRO"
    }

    agora = datetime.now(ZoneInfo("America/Sao_Paulo"))
    return f"{agora.day} DE {meses[agora.month]} DE {agora.year}"


def montar_texto_certificado(nome):
    nome = escapar_html(limpar_texto(nome).upper())
    data_extenso = escapar_html(data_geracao_extenso())

    return (
        f'CERTIFICAMOS QUE <b>{nome}</b> '
        f'PARTICIPOU COMO OUVINTE NO ENCONTRO DE <b>SEMINÁRIOS DE PESQUISA DO PPGE</b>, '
        f'REALIZADO EM <b>14 DE ABRIL DE 2026</b>, '
        f'NA FACULDADE DE CIÊNCIAS E LETRAS DE ARARAQUARA, '
        f'CARACTERIZANDO <b>1 HORA</b> DE ATIVIDADE.<br/><br/>'
        f'ARARAQUARA, <b>{data_extenso}</b>.'
    )


def criar_paragrafo_ajustado(texto):
    tamanho = BASE_FONT
    ultimo_paragrafo = None
    ultima_altura = None

    while tamanho >= MIN_FONT:
        estilo = ParagraphStyle(
            name="certificado",
            fontName=FONTE_NORMAL,
            fontSize=tamanho,
            leading=tamanho * 1.35,
            alignment=TA_CENTER
        )

        paragrafo = Paragraph(texto, estilo)
        _, altura = paragrafo.wrap(TEXT_BOX_WIDTH, TEXT_BOX_HEIGHT)

        ultimo_paragrafo = paragrafo
        ultima_altura = altura

        if altura <= TEXT_BOX_HEIGHT:
            return paragrafo, altura

        tamanho -= 0.5

    return ultimo_paragrafo, ultima_altura


def gerar_pagina_certificado(template_pdf_path, texto):
    template_reader = PdfReader(template_pdf_path)
    base_page = template_reader.pages[0]

    largura = float(base_page.mediabox.width)
    altura = float(base_page.mediabox.height)

    x = (largura - TEXT_BOX_WIDTH) / 2
    y = TEXT_BOX_Y

    buffer = io.BytesIO()
    pdf_canvas = canvas.Canvas(buffer, pagesize=(largura, altura))

    paragrafo, altura_texto = criar_paragrafo_ajustado(texto)

    # desenha o texto centralizado verticalmente dentro da caixa
    paragrafo.drawOn(
        pdf_canvas,
        x,
        y + (TEXT_BOX_HEIGHT - altura_texto) / 2
    )

    pdf_canvas.save()
    buffer.seek(0)

    overlay_reader = PdfReader(buffer)
    overlay_page = overlay_reader.pages[0]

    base_page.merge_page(overlay_page)
    return base_page


def validar_colunas(df, colunas):
    faltantes = [c for c in colunas if c not in df.columns]
    if faltantes:
        raise ValueError(f"Colunas ausentes na planilha: {faltantes}")


# ============================================================
# UPLOAD PLANILHA
# ============================================================

print("Envie a planilha (.xlsx)")
uploaded_xlsx = files.upload()

xlsx_file = next(
    (f for f in uploaded_xlsx if f.lower().endswith(".xlsx")),
    None
)

if not xlsx_file:
    raise FileNotFoundError("Arquivo .xlsx não enviado.")


# ============================================================
# UPLOAD TEMPLATE PDF
# ============================================================

print("Envie o PDF modelo do certificado")
uploaded_pdf = files.upload()

template_pdf = next(
    (f for f in uploaded_pdf if f.lower().endswith(".pdf")),
    None
)

if not template_pdf:
    raise FileNotFoundError("Arquivo .pdf não enviado.")


# ============================================================
# GERAR CERTIFICADOS
# ============================================================

df = pd.read_excel(xlsx_file)
validar_colunas(df, [COL_NOME, COL_EMAIL])

writer = PdfWriter()
emails = []

for _, row in df.iterrows():
    nome = row[COL_NOME]
    email = row[COL_EMAIL]

    texto = montar_texto_certificado(nome)
    pagina = gerar_pagina_certificado(template_pdf, texto)

    writer.add_page(pagina)

    email_limpo = normalizar_email(email)
    if email_limpo:
        emails.append(email_limpo)

with open(OUTPUT_PDF, "wb") as f:
    writer.write(f)

files.download(OUTPUT_PDF)


# ============================================================
# GERAR LISTA DE EMAILS
# ============================================================

emails_unicos = sorted(set(emails))

with open(OUTPUT_EMAILS, "w", encoding="utf-8") as f:
    f.write(", ".join(emails_unicos))

files.download(OUTPUT_EMAILS)


# ============================================================
# RESUMO
# ============================================================

print("Certificados gerados:", len(df))
print("E-mails únicos:", len(emails_unicos))
print("Data:", data_geracao_extenso())

Envie a planilha (.xlsx)


Saving Lista - Ouvintes.xlsx to Lista - Ouvintes (6).xlsx
Envie o PDF modelo do certificado


Saving Modelo certificado.pdf to Modelo certificado (6).pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Certificados gerados: 1
E-mails únicos: 1
Data: 14 DE ABRIL DE 2026
